# Middleware

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

## Summarization Middleware

In [4]:

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model

# Initialize Groq model
llm = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="groq",
    temperature=0.7,
)

# Create agent with message summarization
agent = create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("messages", 10),
            keep=("messages", 4),
        )
    ],
)

# Run the agent
config = {
    "configurable": {
        "thread_id": "conversation-1"
    }
}

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Explain LangChain in simple terms."
            }
        ]
    },
    config=config,
)

print(response["messages"][-1].content)


**LangChain in a nutshell**

Think of LangChain as a *toolbox* that helps you build smart apps with large language models (LLMs) like GPT‑4, Claude, or Llama.  
Instead of writing a lot of boiler‑plate code to:

* ask the model a question,
* feed it text from a database or the web,
* keep track of the conversation history, or
* chain several model calls together (e.g., “first summarize, then answer”),

LangChain gives you ready‑made building blocks that you can snap together.

**Key ideas**

| What it does | Why it matters |
|--------------|----------------|
| **Chaining** – you can link multiple LLM calls or other tools (search engines, calculators, databases) in a sequence. | Lets you create complex workflows (e.g., “search the web, pull relevant facts, then generate a response”). |
| **Memory** – keeps track of past turns in a conversation. | Makes chatbots feel more natural and context‑aware. |
| **Retrieval** – pulls in relevant documents before asking the model. | Improves accura

In [5]:
# run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [7]:
# test data
questions=[
    "Waht is 2+2",
    "Waht is 5*9",
    "Waht is 8/2"
]

for q in questions:
    response = agent.invoke(
        {"messages": [{"role": "user", "content": q}]},
        config=config,
    )
print(f"Message: {response}")
print(f"messages: {len(response['messages'])}")

Message: {'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nProvide answers to basic arithmetic questions asked by the user.\n\n## SUMMARY\n- The user asked for the results of the following expressions:  \n  - 2\u202f+\u202f2 → 4  \n  - 5\u202f×\u202f9 → 45  \n  - 8\u202f÷\u202f2 → 4  \n- The user repeated the question “Waht is 2+2” (likely a typo for “What is 2+2”).\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nAnswer the repeated question: “What is 2+2?”', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='85e2b07f-d274-4041-a443-923df7da18cf'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'User asks again "Waht is 2+2". Should answer 4. There\'s no policy issue.'}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 158, 'total_tokens': 200, 'completion_time': 0.046537512, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time'

## Token Size

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

@tool
def search_hotels(city:str) -> str:
    """Search hotels - return long response to use more tokens."""
    return f"""Hostel in {city}:
    1. Grang Hotel - 5 star, $360/night, spa, pool, gym
    2.City Inn - 4 star, $180/night, buisness center
    3. Budget Stay - 3 star, $75/night, free wifi"""

# Initialize Groq model
llm = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="groq",
    temperature=0.7,
)

# Create agent with message summarization
agent = create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens", 100),
            keep=("tokens", 40),
        )
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token center
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4

In [12]:

cities = ["Paris", "London", "Tokyo", "India", "Dubai"]

for city in cities:
    print(f"\n{'=' * 60}")
    print(f"CITY: {city}")
    print("=" * 60)

    city_config = {
        "configurable": {
            "thread_id": f"hotel-search-{city.lower()}"
        }
    }

    try:
        response = agent.invoke(
            {
                "messages": [
                    HumanMessage(
                        content=f"Find hotels in {city}. Give 3 examples."
                    )
                ]
            },
            config=city_config
        )

        messages = response["messages"]
        tokens = count_tokens(messages)

        print(f"Approximate tokens: {tokens}")
        print(f"Number of messages: {len(messages)}")
        print("\nAgent response:")

        for message in reversed(messages):
            if message.type == "ai" and message.content:
                print(message.content)
                break

    except Exception as e:
        print(f"Error for {city}: {e}")

    print()



CITY: Paris
Approximate tokens: 387
Number of messages: 4

Agent response:
Here are three Parisian hotel options you might consider, each with a quick snapshot of what makes them stand out:

| # | Hotel | Star Rating | Approx. Price / Night | Notable Features |
|---|-------|-------------|-----------------------|------------------|
| 1 | **Grang Hotel** | 5 ★ | ~US $360 | Luxurious spa, indoor pool, full‑service gym, fine‑dining restaurant on‑site. Ideal for a pampered stay. |
| 2 | **City Inn** | 4 ★ | ~US $180 | Modern business center, complimentary Wi‑Fi, rooftop terrace with city views, close to major transit hubs. |
| 3 | **Budget Stay** | 3 ★ | ~US $75 | Affordable, clean rooms, free Wi‑Fi, located near a metro line, great value for budget travelers. |

These examples cover a range of budgets and styles, from upscale luxury to budget‑friendly options. Let me know if you’d like more details or additional recommendations!


CITY: London
Approximate tokens: 274
Number of messages: 4

## Fraction

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

@tool
def search_hotels(city:str) -> str:
    """Search hotels - return long response to use more tokens."""
    return f"""Hostel in {city}:
    1. Grang Hotel - 5 star, $360/night, spa, pool, gym
    2.City Inn - 4 star, $180/night, buisness center
    3. Budget Stay - 3 star, $75/night, free wifi"""

# Initialize Groq model
llm = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="groq",
    temperature=0.7,
)

# Create agent with message summarization
agent = create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("fraction", 0.005),
            keep=("fraction", 0.002),
        )
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token center
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4



cities = ["Paris", "London", "Tokyo", "India", "Dubai"]

for city in cities:
    print(f"\n{'=' * 60}")
    print(f"CITY: {city}")
    print("=" * 60)

    city_config = {
        "configurable": {
            "thread_id": f"hotel-search-{city.lower()}"
        }
    }

    try:
        response = agent.invoke(
            {
                "messages": [
                    HumanMessage(
                        content=f"Find hotels in {city}. Give 3 examples."
                    )
                ]
            },
            config=city_config
        )

        messages = response["messages"]
        tokens = count_tokens(messages)

        print(f"Approximate tokens: {tokens}")
        print(f"Number of messages: {len(messages)}")
        print("\nAgent response:")

        for message in reversed(messages):
            if message.type == "ai" and message.content:
                print(message.content)
                break

    except Exception as e:
        print(f"Error for {city}: {e}")

    print()



CITY: Paris
Approximate tokens: 202
Number of messages: 4

Agent response:
Here are three hotels you might consider when visiting Paris:

| Hotel | Star Rating | Approx. Price (per night) | Highlights |
|-------|-------------|---------------------------|------------|
| **Grang Hotel** | 5★ | $360 | Luxury suites, spa, indoor pool, fitness center |
| **City Inn** | 4★ | $180 | Business‑friendly rooms, meeting rooms, on‑site café |
| **Budget Stay** | 3★ | $75 | Cozy rooms, free Wi‑Fi, convenient location |

Feel free to let me know if you’d like more details (e.g., address, phone number, or availability) or if you’re interested in a different type of accommodation!


CITY: London
Approximate tokens: 212
Number of messages: 4

Agent response:
Here are three London hotels that fit a range of budgets and amenities:

| # | Hotel | Star Rating | Approx. Price (per night) | Key Amenities |
|---|-------|-------------|---------------------------|---------------|
| 1 | **Grang Hotel** | 5 ★ | *